# DB And Migrations Walkthrough

This notebook shows how the live runtime database gets initialized, how to inspect the applied migrations, and how safety snapshots fit into the workflow.


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Start Jupyter from somewhere inside the repo.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
FIXTURES_DIR = REPO_ROOT / "tests" / "fixtures"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def reset_dir(path: Path) -> Path:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def copy_fixture_bundle(destination: Path, fixture_name: str, *filenames: str) -> dict[str, Path]:
    destination.mkdir(parents=True, exist_ok=True)
    copied: dict[str, Path] = {}
    for filename in filenames:
        source = FIXTURES_DIR / fixture_name / filename
        target = destination / filename
        shutil.copyfile(source, target)
        copied[filename] = target
    return copied

from mtg_source_stack.db.connection import connect
from mtg_source_stack.db.schema import initialize_database, require_current_schema
from mtg_source_stack.db.snapshots import create_database_snapshot, list_database_snapshots

WORKDIR = reset_dir(REPO_ROOT / "var" / "walkthrough" / "01_db_and_migrations")
DB_PATH = WORKDIR / "collection.db"
print(f"Working directory: {WORKDIR}")
print(f"Database path: {DB_PATH}")


## Initialize A Fresh Database

`initialize_database()` is the main entrypoint for bringing a local SQLite file up to the current schema version.


In [ ]:
initialize_database(DB_PATH)
require_current_schema(DB_PATH)
print(f"Database exists: {DB_PATH.exists()}")
print(f"Database size: {DB_PATH.stat().st_size} bytes")


## Inspect Tables And Applied Migrations

The current runtime model is the MVP schema plus the migration history recorded in `schema_migrations`.


In [ ]:
with connect(DB_PATH) as connection:
    tables = [row["name"] for row in connection.execute(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
    ).fetchall()]
    migrations = [dict(row) for row in connection.execute(
        "SELECT version, name, applied_at FROM schema_migrations ORDER BY version"
    ).fetchall()]

print("Tables:")
for table in tables:
    print(f"  - {table}")

print("\nApplied migrations:")
for migration in migrations:
    print(migration)


## Compare Runtime Schema And Docs Copy

The runtime schema file is canonical. The docs-side copy should stay in sync for convenient reading.


In [ ]:
runtime_schema = (REPO_ROOT / "src" / "mtg_source_stack" / "mtg_mvp_schema.sql").read_text(encoding="utf-8")
docs_schema = (REPO_ROOT / "docs" / "schema_mvp.sql").read_text(encoding="utf-8")
print(f"Schema files match: {runtime_schema == docs_schema}")


## Create And Inspect A Safety Snapshot

Snapshots are coarse-grained safety copies of the database file. The CLI creates them automatically before higher-risk writes, and you can also create them directly.


In [ ]:
snapshot = create_database_snapshot(DB_PATH, label="after_notebook_init")
snapshots = list_database_snapshots(DB_PATH)
print(json.dumps(snapshot, indent=2))
print("\nKnown snapshots:")
for item in snapshots:
    print(f"  - {item['snapshot_name']} ({item['label']})")


## Next Step

Open `02_importer_walkthrough.ipynb` to see how Scryfall and MTGJSON bulk files populate the local catalog and pricing tables.
